In [ ]:
# ============================================================
# Cell 1: Install required packages
# ============================================================
!pip install opacus -q

In [ ]:
# ============================================================
# Cell 2: Imports
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
import random
import pandas as pd
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import warnings
import time

warnings.filterwarnings('ignore')

# Check Opacus import
try:
    from opacus import PrivacyEngine
    print("Opacus imported successfully.")
except ImportError:
    print("ERROR: Opacus not found. Run Cell 1 to install.")
    raise

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# ============================================================
# Cell 3: Seed configuration and bootstrap confidence interval
# ============================================================
# 10 seeds as specified in Section 4.2 of the paper
SEEDS = [42, 123, 456, 789, 1011, 2022, 3033, 4044, 5055, 6066]


def set_seed(seed=42):
    """Set random seed for reproducibility across all libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def bootstrap_ci(values, n_bootstrap=10000, confidence=0.95):
    """
    Compute bootstrap 95% confidence interval for the mean.
    Uses 10,000 resamples as stated in Section 4.2.
    """
    values = np.array(values)
    boot_means = np.empty(n_bootstrap)
    for i in range(n_bootstrap):
        sample = np.random.choice(values, size=len(values), replace=True)
        boot_means[i] = np.mean(sample)
    lower = np.percentile(boot_means, (1 - confidence) / 2 * 100)
    upper = np.percentile(boot_means, (1 + confidence) / 2 * 100)
    return np.mean(values), lower, upper


print(f"Configured {len(SEEDS)} seeds: {SEEDS}")

In [ ]:
# ============================================================
# Cell 4: Granular Ball data structure
# ============================================================
class GranularBall:
    """
    Represents a single granular ball characterized by:
      - center:  d-dimensional mean of constituent points
      - radius:  max distance from center to any point in the ball
      - label:   majority class
      - purity:  fraction of points belonging to the majority class
      - size:    number of constituent points
    """
    def __init__(self, center, radius, label, purity, size):
        self.center = np.array(center, dtype=np.float64)
        self.radius = float(radius)
        self.label = int(label)
        self.purity = float(purity)
        self.size = int(size)

    def __repr__(self):
        return (f"GranularBall(dim={len(self.center)}, r={self.radius:.4f}, "
                f"label={self.label}, purity={self.purity:.3f}, n={self.size})")

In [ ]:
# ============================================================
# Cell 5: Granular Ball Generation Algorithm (Algorithm 1)
# ============================================================
def _make_terminal_ball(X, y):
    """Create a terminal granular ball from data points."""
    center = np.mean(X, axis=0)
    distances = np.linalg.norm(X - center, axis=1)
    radius = float(np.max(distances)) if len(distances) > 0 else 0.0

    unique, counts = np.unique(y, return_counts=True)
    majority_idx = np.argmax(counts)
    label = unique[majority_idx]
    purity = float(counts[majority_idx] / len(y))

    return [GranularBall(center, radius, label, purity, len(y))]


def generate_granular_balls(X, y, theta=0.1, min_samples=800,
                            max_depth=50, depth=0):
    """
    Recursively partition data into granular balls (Algorithm 1).

    Stopping conditions (any one triggers a terminal ball):
      1. depth >= max_depth
      2. |X| < min_samples
      3. max variance across dimensions <= theta

    Splits along the dimension of maximum variance at the median.
    """
    n, d = X.shape

    # --- Stopping conditions ---
    if depth >= max_depth or n < min_samples:
        return _make_terminal_ball(X, y)

    var_per_dim = np.var(X, axis=0)
    if np.max(var_per_dim) <= theta:
        return _make_terminal_ball(X, y)

    # --- Find split dimension and threshold ---
    split_dim = np.argmax(var_per_dim)
    median_val = np.median(X[:, split_dim])

    left_mask = X[:, split_dim] <= median_val
    right_mask = ~left_mask

    # Guard against degenerate split
    if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
        return _make_terminal_ball(X, y)

    # --- Recursive splitting ---
    left_balls = generate_granular_balls(
        X[left_mask], y[left_mask],
        theta, min_samples, max_depth, depth + 1
    )
    right_balls = generate_granular_balls(
        X[right_mask], y[right_mask],
        theta, min_samples, max_depth, depth + 1
    )

    return left_balls + right_balls


# ---- Quick sanity check ----
set_seed(42)
X_demo, y_demo = make_moons(n_samples=100, noise=0.1, random_state=42)
balls_demo = generate_granular_balls(X_demo, y_demo,
                                     theta=0.01, min_samples=20, max_depth=10)
print(f"Sanity check: {len(balls_demo)} balls from 100 Two-Moons samples")
for b in balls_demo:
    print(f"  {b}")

In [ ]:
# ============================================================
# Cell 6: Differential Privacy Noise Addition (Corrected)
# ============================================================
def apply_dp_noise(balls, epsilon_total, delta_total):
    """
    Add DP noise to each granular ball.
    For each ball we add:
      - Gaussian noise to centre (L2 sensitivity)
      - Laplace noise to radius (L1 sensitivity)
    Privacy budget is split between centre and radius per ball:
        epsilon_ball = epsilon_total / (2 * k)   for centre
        epsilon_ball = epsilon_total / (2 * k)   for radius
    where k = number of balls.
    """
    import numpy as np
    k = len(balls)
    if k == 0:
        return balls

    # Per-mechanism budget (centre and radius are independent)
    epsilon_per_mechanism = epsilon_total / (2 * k)
    # Delta is split equally (we use delta_total/(2k) for each mechanism)
    delta_per_mechanism = delta_total / (2 * k)

    private_balls = []
    for ball in balls:
        d = len(ball.center)
        n = ball.size

        # Sensitivity for centre (L2) and radius (L1) – same formula
        sensitivity = 2 * np.sqrt(d) / n

        # 1. Gaussian noise for centre
        sigma = sensitivity * np.sqrt(2 * np.log(1.25 / delta_per_mechanism)) / epsilon_per_mechanism
        noise_center = np.random.normal(0, sigma, size=d)
        new_center = ball.center + noise_center

        # 2. Laplace noise for radius
        scale = sensitivity / epsilon_per_mechanism
        noise_radius = np.random.laplace(0, scale)
        new_radius = max(0, ball.radius + noise_radius)

        private_balls.append(GranularBall(
            center=new_center,
            radius=new_radius,
            label=ball.label,
            purity=ball.purity,
            size=ball.size
        ))
    return private_balls

In [ ]:
# # ============================================================
# # Cell 6: DP Noise Addition to Granular Balls (Algorithm 2)
# # ============================================================
# def apply_dp_noise(balls, epsilon_total, delta_total):
#     """
#     Privatize granular balls using:
#       - Gaussian mechanism for centers
#       - Laplace mechanism for radii

#     Sensitivity (as derived in Section 3.2):
#         Δ = 2√d / n   (assumes data normalised to [-1, 1])

#     Sequential composition: ε_ball = ε_total / k,  δ_ball = δ_total / k
#     """
#     k = len(balls)
#     if k == 0:
#         return []

#     epsilon_ball = epsilon_total / k
#     delta_ball = delta_total / k

#     privatized = []
#     for ball in balls:
#         d = len(ball.center)
#         n = ball.size

#         # Sensitivity
#         delta_sens = 2.0 * np.sqrt(d) / n

#         # --- Gaussian noise for center ---
#         # σ = Δ · √(2·ln(1.25/δ_ball)) / ε_ball
#         sigma = (delta_sens
#                  * np.sqrt(2.0 * np.log(1.25 / delta_ball))
#                  / epsilon_ball)
#         noise_center = np.random.normal(0.0, sigma, size=d)
#         private_center = ball.center + noise_center

#         # --- Laplace noise for radius ---
#         # b = Δ / ε_ball
#         b = delta_sens / epsilon_ball
#         noise_radius = np.random.laplace(0.0, b)
#         private_radius = max(0.0, ball.radius + noise_radius)

#         privatized.append(GranularBall(
#             center=private_center,
#             radius=private_radius,
#             label=ball.label,
#             purity=ball.purity,
#             size=ball.size
#         ))

#     return privatized


# # ---- Quick sanity check ----
# set_seed(42)
# private_demo = apply_dp_noise(balls_demo, epsilon_total=5.0, delta_total=1e-5)
# print("Sanity check — before vs after DP noise (ε=5.0):")
# for b_orig, b_priv in zip(balls_demo, private_demo):
#     print(f"  center[:2]: {b_orig.center[:2].round(4)} → {b_priv.center[:2].round(4)}")
#     print(f"  radius:     {b_orig.radius:.4f} → {b_priv.radius:.4f}")

In [ ]:
# ============================================================
# Cell 7: Prediction with Temperature Scaling (Section 3.3)
# ============================================================
def predict_dpgbc(balls, X, tau=0.5):
    """
    Distance-weighted voting with temperature parameter τ.
    w_i(x) = p_i · exp(-‖x - c̃_i‖ / τ)
    P(y=c|x) = Σ_{i:label_i=c} w_i / Σ_j w_j
    ŷ = argmax_c P(y=c|x)
    """
    predictions = np.empty(len(X), dtype=int)
    probabilities = []

    for idx, x in enumerate(X):
        weights = np.empty(len(balls))
        labels = np.empty(len(balls), dtype=int)

        for j, ball in enumerate(balls):
            dist = np.linalg.norm(x - ball.center)
            weights[j] = ball.purity * np.exp(-dist / tau)
            labels[j] = ball.label

        # Class probabilities
        unique_labels = np.unique(labels)
        probs = {}
        for c in unique_labels:
            probs[c] = np.sum(weights[labels == c])

        total = sum(probs.values())
        if total > 1e-15:
            for c in probs:
                probs[c] /= total
        else:
            for c in unique_labels:
                probs[c] = 1.0 / len(unique_labels)

        predictions[idx] = max(probs, key=probs.get)
        probabilities.append(probs)

    return predictions, probabilities


def evaluate_dpgbc(balls, X, y, tau=0.5):
    """Return (accuracy, predictions, probabilities)."""
    preds, probs = predict_dpgbc(balls, X, tau)
    acc = float(np.mean(preds == y))
    return acc, preds, probs

In [ ]:
# ============================================================
# Cell 8: End-to-End DP-GbC (Algorithm 3)
# ============================================================
def run_dpgbc(X_pub, y_pub, X_test, y_test, epsilon, delta,
              theta=0.1, min_samples=800, max_depth=50, tau=0.5):
    """
    Phase 1: Generate granular balls on public data
    Phase 2: Apply differential privacy noise
    Phase 3: Evaluate on test data
    """
    # Phase 1
    balls = generate_granular_balls(X_pub, y_pub,
                                    theta, min_samples, max_depth)
    k = len(balls)

    # Phase 2
    private_balls = apply_dp_noise(balls, epsilon, delta)

    # Phase 3
    acc, preds, probs = evaluate_dpgbc(private_balls, X_test, y_test, tau)

    return private_balls, acc, preds, probs, k

In [ ]:
# ============================================================
# Cell 9: Neural Network Architecture (Section 3.4)
# ============================================================
class SimpleNet(nn.Module):
    """
    2-layer fully connected network as described in Section 3.4:
      input_dim → 256 → num_classes
    """
    def __init__(self, input_dim, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
# ============================================================
# Cell 10: DP-SGD Training with Opacus (Algorithm 4)
# ============================================================
def train_dpsgd(X_train, y_train, X_test, y_test,
                target_epsilon=5.0, delta=1e-5,
                batch_size=256, epochs=4, lr=0.1, momentum=0.9,
                max_grad_norm=1.0, seed=42):
    """
    Train SimpleNet with DP-SGD via Opacus.
    Privacy accounting uses Rényi DP (RDP).

    Returns: (test_accuracy, actual_epsilon, trained_model)
    """
    set_seed(seed)

    input_dim = X_train.shape[1]
    num_classes = int(np.max(y_train)) + 1

    # Tensors
    X_train_t = torch.FloatTensor(X_train)
    y_train_t = torch.LongTensor(y_train.astype(np.int64))
    X_test_t = torch.FloatTensor(X_test)
    y_test_t = torch.LongTensor(y_test.astype(np.int64))

    train_ds = TensorDataset(X_train_t, y_train_t)
    test_ds = TensorDataset(X_test_t, y_test_t)

    # drop_last ensures consistent batch size for privacy accounting
    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True, drop_last=True)
    test_loader = DataLoader(test_ds, batch_size=512, shuffle=False)

    # Model & optimizer
    model = SimpleNet(input_dim, num_classes)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum)

    # Attach Opacus PrivacyEngine
    privacy_engine = PrivacyEngine(secure_mode=False)
    model, optimizer, train_loader = privacy_engine.make_private_with_epsilon(
        module=model,
        optimizer=optimizer,
        data_loader=train_loader,
        epochs=epochs,
        target_epsilon=target_epsilon,
        target_delta=delta,
        max_grad_norm=max_grad_norm,
    )

    # Training loop
    model.train()
    for epoch in range(epochs):
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad()
            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

    # Evaluation
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            logits = model(x_batch)
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.numpy())
            all_labels.extend(y_batch.numpy())

    test_acc = float(accuracy_score(all_labels, all_preds))
    actual_epsilon = privacy_engine.get_epsilon(delta=delta)

    return test_acc, actual_epsilon, model

In [ ]:
# ============================================================
# Cell 11: Membership Inference Attack Evaluation (Section 3.5)
# ============================================================
def compute_losses_dpgbc(balls, X, y, tau=0.5):
    """Negative log-probability loss for DP-GbC."""
    _, probs = predict_dpgbc(balls, X, tau)
    losses = np.empty(len(X))
    for i, p in enumerate(probs):
        prob = p.get(y[i], 1e-10)
        losses[i] = -np.log(max(prob, 1e-10))
    return losses


def compute_losses_dpsgd(model, X, y):
    """Cross-entropy loss for DP-SGD model."""
    model.eval()
    X_t = torch.FloatTensor(X)
    y_t = torch.LongTensor(y.astype(np.int64))
    criterion = nn.CrossEntropyLoss(reduction='none')
    with torch.no_grad():
        logits = model(X_t)
        losses = criterion(logits, y_t).numpy()
    return losses


def run_mia(train_losses, test_losses):
    """
    Loss-based MIA (Shokri et al.).
    Logistic regression distinguishes train vs test losses.
    50 % = random guess → strong privacy.
    """
    train_labels = np.ones(len(train_losses))
    test_labels = np.zeros(len(test_losses))

    all_losses = np.concatenate([train_losses, test_losses]).reshape(-1, 1)
    all_labels = np.concatenate([train_labels, test_labels])

    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(all_losses, all_labels)
    preds = clf.predict(all_losses)
    return float(accuracy_score(all_labels, preds))

In [ ]:
# ============================================================
# Cell 12: CIFAR-10 + ResNet-18 Feature Extraction (Section 4.1)
# ============================================================
def _extract_features(loader, feature_extractor):
    """Pass data through ResNet-18 (up to avgpool) → 512-D vectors."""
    feature_extractor.eval()
    feats, labels = [], []
    with torch.no_grad():
        for images, lbls in loader:
            out = feature_extractor(images)            # (B, 512, 1, 1)
            out = out.view(out.size(0), -1)           # (B, 512)
            feats.append(out.numpy())
            labels.extend(lbls.numpy())
    return np.concatenate(feats, axis=0), np.array(labels)


def load_cifar10_features(n_train_samples=5000, seed=42):
    """
    1. Download CIFAR-10 (32×32 images, 10 classes)
    2. Extract 512-D features via pretrained ResNet-18
       (remove FC layer; avgpool output is 512×1×1 → flatten to 512)
    3. Subsample n_train_samples from training set
    4. Split: 1000 public + 4000 private
    5. Normalise to [-1,1] using min/max from PRIVATE set only
    """
    set_seed(seed)
    print("Downloading CIFAR-10 …")

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])

    train_ds = torchvision.datasets.CIFAR10(root='./data', train=True,
                                           download=True, transform=transform)
    test_ds  = torchvision.datasets.CIFAR10(root='./data', train=False,
                                           download=True, transform=transform)

    train_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False)

    # Pretrained ResNet-18 (remove FC, keep avgpool)
    print("Loading pretrained ResNet-18 …")
    try:
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    except TypeError:
        resnet = models.resnet18(pretrained=True)
    feature_extractor = nn.Sequential(*list(resnet.children())[:-1])
    feature_extractor.eval()

    print("Extracting training features …")
    train_feats, train_labels = _extract_features(train_loader, feature_extractor)
    print("Extracting test features …")
    test_feats, test_labels = _extract_features(test_loader, feature_extractor)

    print(f"Feature dimensionality: {train_feats.shape[1]}")

    # Subsample training set
    idx = np.random.choice(len(train_feats), n_train_samples, replace=False)
    X_sub, y_sub = train_feats[idx], train_labels[idx]

    # Public / private split
    X_pub, X_priv, y_pub, y_priv = train_test_split(
        X_sub, y_sub, test_size=0.8, random_state=seed)

    # Normalise to [-1, 1] using PRIVATE min/max only
    fmin = X_priv.min(axis=0)
    fmax = X_priv.max(axis=0)
    frange = fmax - fmin
    frange[frange == 0] = 1.0

    X_priv_n = np.clip(2.0 * (X_priv - fmin) / frange - 1.0, -1, 1)
    X_pub_n  = np.clip(2.0 * (X_pub  - fmin) / frange - 1.0, -1, 1)
    X_test_n = np.clip(2.0 * (test_feats - fmin) / frange - 1.0, -1, 1)

    print(f"Public:  {X_pub_n.shape}   Private: {X_priv_n.shape}   "
          f"Test: {X_test_n.shape}")
    return X_pub_n, y_pub, X_priv_n, y_priv, X_test_n, test_labels

In [ ]:
# ============================================================
# Cell 13: Load Two Moons, MNIST, UCI Adult (Section 4.1)
# ============================================================

def load_two_moons(n_samples=2000, noise=0.1, seed=42):
    """2-D synthetic binary classification."""
    set_seed(seed)
    X, y = make_moons(n_samples=n_samples, noise=noise, random_state=seed)
    # 1000 public, 500 private, 500 test
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25,
                                               random_state=seed)
    X_pub, X_priv, y_pub, y_priv = train_test_split(X_tr, y_tr,
                                                     test_size=0.5,
                                                     random_state=seed)
    print(f"Two Moons — Public: {X_pub.shape}  Private: {X_priv.shape}  "
          f"Test: {X_te.shape}")
    return X_pub, y_pub, X_priv, y_priv, X_te, y_te


def load_mnist(n_train_samples=5000, seed=42):
    """784-D flattened MNIST, normalised to [0, 1]."""
    set_seed(seed)
    print("Downloading MNIST …")
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.view(-1).numpy())
    ])
    train_ds = torchvision.datasets.MNIST(root='./data', train=True,
                                         download=True, transform=transform)
    test_ds  = torchvision.datasets.MNIST(root='./data', train=False,
                                         download=True, transform=transform)

    X_train_full = np.stack([train_ds[i][0] for i in range(len(train_ds))])
    y_train_full = np.array([train_ds[i][1] for i in range(len(train_ds))])
    X_test = np.stack([test_ds[i][0] for i in range(len(test_ds))])
    y_test = np.array([test_ds[i][1] for i in range(len(test_ds))])

    idx = np.random.choice(len(X_train_full), n_train_samples, replace=False)
    X_sub, y_sub = X_train_full[idx], y_train_full[idx]

    X_pub, X_priv, y_pub, y_priv = train_test_split(
        X_sub, y_sub, test_size=0.8, random_state=seed)

    print(f"MNIST — Public: {X_pub.shape}  Private: {X_priv.shape}  "
          f"Test: {X_test.shape}")
    return X_pub, y_pub, X_priv, y_priv, X_test, y_test


def load_adult(seed=42):
    """UCI Adult — ~100-D after one-hot encoding, binary classification."""
    set_seed(seed)
    print("Downloading UCI Adult …")
    columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num',
               'marital-status', 'occupation', 'relationship', 'race', 'sex',
               'capital-gain', 'capital-loss', 'hours-per-week',
               'native-country', 'income']

    url1 = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
    url2 = ("https://raw.githubusercontent.com/ageron/handson-ml2/master/"
            "datasets/adult/adult.data")
    try:
        df = pd.read_csv(url1, names=columns, na_values=' ?',
                         skipinitialspace=True)
    except Exception:
        df = pd.read_csv(url2, names=columns, na_values=' ?',
                         skipinitialspace=True)

    df = df.dropna()
    print(f"Adult — {len(df)} rows after dropping missing values")

    y = (df['income'] == '>50K').astype(int).values
    X_df = df.drop('income', axis=1)
    X_enc = pd.get_dummies(X_df, drop_first=False).astype(np.float32)

    # Normalise numerical columns to [-1, 1]
    num_cols = ['age', 'fnlwgt', 'education-num',
                'capital-gain', 'capital-loss', 'hours-per-week']
    num_cols = [c for c in num_cols if c in X_enc.columns]
    for c in num_cols:
        cmin, cmax = X_enc[c].min(), X_enc[c].max()
        rng = cmax - cmin
        if rng > 0:
            X_enc[c] = 2.0 * (X_enc[c] - cmin) / rng - 1.0

    X = X_enc.values
    print(f"Adult — feature dim = {X.shape[1]}")

    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                               random_state=seed)
    X_pub, X_priv, y_pub, y_priv = train_test_split(X_tr, y_tr,
                                                     test_size=0.8,
                                                     random_state=seed)
    print(f"Adult — Public: {X_pub.shape}  Private: {X_priv.shape}  "
          f"Test: {X_te.shape}")
    return X_pub, y_pub, X_priv, y_priv, X_te, y_te

In [ ]:
# ============================================================
# Cell 14: TABLE 1 — Main Comparison on CIFAR-10 (Section 5.1)
# ============================================================
print("=" * 72)
print("TABLE 1: Main Comparison on CIFAR-10 (512-D ResNet Features, ε=5.0)")
print("=" * 72)

EPSILON = 5.0
DELTA   = 1e-5
TAU     = 0.5
N_SEEDS = 10          # full 10-seed evaluation

X_pub, y_pub, X_priv, y_priv, X_test, y_test = load_cifar10_features(
    n_train_samples=5000, seed=42)

# ---- DP-GbC coarse (m=800 → ~2 balls) ----
print(f"\n>>> DP-GbC  θ=0.1, m=800  (ε={EPSILON}, τ={TAU})")
accs_coarse = []
for s in SEEDS[:N_SEEDS]:
    set_seed(s)
    _, acc, _, _, k = run_dpgbc(X_pub, y_pub, X_test, y_test,
                                EPSILON, DELTA,
                                theta=0.1, min_samples=800,
                                max_depth=50, tau=TAU)
    accs_coarse.append(acc)
    print(f"  seed {s:>5d}:  acc={acc*100:.2f}%  (k={k} balls)")

m, lo, hi = bootstrap_ci(accs_coarse)
print(f"  >>> MEAN = {m*100:.2f}%   95% CI = [{lo*100:.2f}%, {hi*100:.2f}%]")

# ---- DP-GbC fine (m=50 → many balls) ----
print(f"\n>>> DP-GbC  θ=0.1, m=50  (ε={EPSILON}, τ={TAU})")
accs_fine = []
for s in SEEDS[:N_SEEDS]:
    set_seed(s)
    _, acc, _, _, k = run_dpgbc(X_pub, y_pub, X_test, y_test,
                                EPSILON, DELTA,
                                theta=0.1, min_samples=50,
                                max_depth=50, tau=TAU)
    accs_fine.append(acc)
    print(f"  seed {s:>5d}:  acc={acc*100:.2f}%  (k={k} balls)")

m, lo, hi = bootstrap_ci(accs_fine)
print(f"  >>> MEAN = {m*100:.2f}%   95% CI = [{lo*100:.2f}%, {hi*100:.2f}%]")

# ---- DP-SGD ----
print(f"\n>>> DP-SGD  target ε={EPSILON}")
accs_sgd, eps_sgd = [], []
for s in SEEDS[:N_SEEDS]:
    acc, eps_act, _ = train_dpsgd(X_priv, y_priv, X_test, y_test,
                                  target_epsilon=EPSILON, delta=DELTA,
                                  batch_size=256, epochs=4,
                                  lr=0.1, momentum=0.9,
                                  max_grad_norm=1.0, seed=s)
    accs_sgd.append(acc)
    eps_sgd.append(eps_act)
    print(f"  seed {s:>5d}:  acc={acc*100:.2f}%  ε_actual={eps_act:.2f}")

m, lo, hi = bootstrap_ci(accs_sgd)
print(f"  >>> MEAN = {m*100:.2f}%   95% CI = [{lo*100:.2f}%, {hi*100:.2f}%]")
print(f"  >>> Mean actual ε = {np.mean(eps_sgd):.2f}")

# ---- Summary ----
print("\n" + "=" * 72)
print("TABLE 1 SUMMARY")
print("=" * 72)
print(f"{'Method':<42} {'Acc (%)':<10} {'95% CI':<22} {'ε':<8}")
print("-" * 82)
m, lo, hi = bootstrap_ci(accs_coarse)
print(f"{'DP-GbC (θ=0.1, m=800)':<42} {m*100:<10.2f} "
      f"[{lo*100:.2f}%, {hi*100:.2f}%]   {EPSILON:.2f}")
m, lo, hi = bootstrap_ci(accs_fine)
print(f"{'DP-GbC (θ=0.1, m=50)':<42} {m*100:<10.2f} "
      f"[{lo*100:.2f}%, {hi*100:.2f}%]   {EPSILON:.2f}")
m, lo, hi = bootstrap_ci(accs_sgd)
print(f"{'DP-SGD (target ε=5.0)':<42} {m*100:<10.2f} "
      f"[{lo*100:.2f}%, {hi*100:.2f}%]   {np.mean(eps_sgd):.2f}")

In [ ]:
# ============================================================
# Cell 15: TABLE 2 — Temperature Tuning (Section 5.2)
# ============================================================
print("=" * 60)
print("TABLE 2: Temperature Tuning (public set, clean balls, m=400)")
print("=" * 60)

TAU_VALUES = [0.5, 1.0, 2.0, 5.0, 10.0]
EPSILON = 5.0
DELTA   = 1e-5

set_seed(42)
balls_clean = generate_granular_balls(X_pub, y_pub,
                                     theta=0.1, min_samples=400,
                                     max_depth=50)
print(f"Number of balls (m=400): {len(balls_clean)}\n")

print(f"{'τ':<8} {'Public Acc (%)':<18}")
print("-" * 28)
for tau in TAU_VALUES:
    acc, _, _ = evaluate_dpgbc(balls_clean, X_pub, y_pub, tau=tau)   # <-- CHANGED
    print(f"{tau:<8} {acc*100:<18.2f}")

In [ ]:
# ============================================================
# Cell 16: Epsilon Sweep Analysis (Section 5.3)
# ============================================================
print("=" * 70)
print("EPSILON SWEEP: DP-GbC vs DP-SGD on CIFAR-10 (m=400, τ=0.5)")
print("=" * 70)

EPSILONS = [0.5, 1.0, 2.0, 5.0, 10.0]
DELTA    = 1e-5
TAU      = 0.5
N_SEEDS_SWEEP = 3   # reduced seeds for sweep (increase for full eval)

dpgbc_sweep = []
dpsgd_sweep = []

for eps in EPSILONS:
    print(f"\n--- ε = {eps} ---")

    # DP-GbC
    accs_gbc = []
    for s in SEEDS[:N_SEEDS_SWEEP]:
        set_seed(s)
        _, acc, _, _, k = run_dpgbc(X_pub, y_pub, X_test, y_test,
                                    eps, DELTA,
                                    theta=0.1, min_samples=400,
                                    max_depth=50, tau=TAU)
        accs_gbc.append(acc)
    m_gbc = np.mean(accs_gbc)
    dpgbc_sweep.append(m_gbc)
    print(f"  DP-GbC:  {m_gbc*100:.2f}%  (seeds: "
          f"{[f'{a*100:.1f}' for a in accs_gbc]})")

    # DP-SGD
    accs_sgd = []
    for s in SEEDS[:N_SEEDS_SWEEP]:
        acc, _, _ = train_dpsgd(X_priv, y_priv, X_test, y_test,
                                target_epsilon=eps, delta=DELTA,
                                batch_size=256, epochs=4,
                                lr=0.1, momentum=0.9,
                                max_grad_norm=1.0, seed=s)
        accs_sgd.append(acc)
    m_sgd = np.mean(accs_sgd)
    dpsgd_sweep.append(m_sgd)
    print(f"  DP-SGD:  {m_sgd*100:.2f}%  (seeds: "
          f"{[f'{a*100:.1f}' for a in accs_sgd]})")

# Plot
plt.figure(figsize=(8, 5))
plt.plot(EPSILONS, [a * 100 for a in dpgbc_sweep],
         'o-', label='DP-GbC (m=400)', linewidth=2, markersize=8)
plt.plot(EPSILONS, [a * 100 for a in dpsgd_sweep],
         's--', label='DP-SGD', linewidth=2, markersize=8)
plt.xscale('log')
plt.xlabel('Privacy Budget ε (log scale)', fontsize=12)
plt.ylabel('Test Accuracy (%)', fontsize=12)
plt.title('Epsilon Sweep: DP-GbC vs DP-SGD on CIFAR-10 Features', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nSweep summary:")
for eps, ag, asg in zip(EPSILONS, dpgbc_sweep, dpsgd_sweep):
    print(f"  ε={eps:<6.1f}  DP-GbC={ag*100:.2f}%  DP-SGD={asg*100:.2f}%")

In [ ]:
# ============================================================
# Cell 17: TABLE 3 — Membership Inference Attack (Section 5.4)
# ============================================================
print("=" * 60)
print("TABLE 3: Membership Inference Attack Success Rates")
print("=" * 60)

set_seed(42)

# --- Non-private GBC (no DP noise) ---
balls_nopriv = generate_granular_balls(X_pub, y_pub,
                                      theta=0.1, min_samples=800,
                                      max_depth=50)
# Split private data into two halves for MIA
X_half1, X_half2, y_half1, y_half2 = train_test_split(
    X_priv, y_priv, test_size=0.5, random_state=42)

loss_train_np = compute_losses_dpgbc(balls_nopriv, X_half1, y_half1, tau=0.5)
loss_test_np  = compute_losses_dpgbc(balls_nopriv, X_half2, y_half2, tau=0.5)
mia_nopriv = run_mia(loss_train_np, loss_test_np)
print(f"Non-private GBC:           MIA = {mia_nopriv*100:.2f}%")

# --- DP-GbC (ε=5.0) ---
balls_dp = apply_dp_noise(balls_nopriv, epsilon_total=5.0, delta_total=1e-5)
loss_train_dp = compute_losses_dpgbc(balls_dp, X_half1, y_half1, tau=0.5)
loss_test_dp  = compute_losses_dpgbc(balls_dp, X_half2, y_half2, tau=0.5)
mia_dpgbc = run_mia(loss_train_dp, loss_test_dp)
print(f"DP-GbC (ε=5.0):            MIA = {mia_dpgbc*100:.2f}%")

# --- DP-SGD (ε≈5.0) ---
_, _, model_sgd = train_dpsgd(X_priv, y_priv, X_test, y_test,
                              target_epsilon=5.0, delta=1e-5,
                              batch_size=256, epochs=4,
                              lr=0.1, momentum=0.9,
                              max_grad_norm=1.0, seed=42)
loss_train_sgd = compute_losses_dpsgd(model_sgd, X_half1, y_half1)
loss_test_sgd  = compute_losses_dpsgd(model_sgd, X_half2, y_half2)
mia_dpsgd = run_mia(loss_train_sgd, loss_test_sgd)
print(f"DP-SGD (ε≈5.0):            MIA = {mia_dpsgd*100:.2f}%")

print(f"\nChance level = 50.00%")
print("-" * 60)
print(f"{'Method':<30} {'MIA Success (%)':<18}")
print("-" * 60)
print(f"{'Non-private GBC':<30} {mia_nopriv*100:<18.2f}")
print(f"{'DP-GbC (ε=5.0)':<30} {mia_dpgbc*100:<18.2f}")
print(f"{'DP-SGD (ε≈5.0)':<30} {mia_dpsgd*100:<18.2f}")

In [ ]:
# ============================================================
# Cell 18: TABLE 4 — Cross-Dataset Comparison (Section 5.5)
# FIX: Use local variables to prevent namespace collision
# ============================================================
print("=" * 72)
print("TABLE 4: Cross-Dataset Comparison (ε=5.0, δ=1e-5)")
print("=" * 72)

EPSILON = 5.0
DELTA   = 1e-5
TAU     = 0.5
N_SEEDS_DS = 3   # use 3 seeds per dataset (increase to 10 for full eval)

datasets = {}

# --- Two Moons (2-D) ---
xp, yp, xpr, ypr, xt, yt = load_two_moons(n_samples=2000, noise=0.1, seed=42)
datasets['Two Moons'] = (xp, yp, xpr, ypr, xt, yt, 2)

# --- UCI Adult (~100-D) ---
xp, yp, xpr, ypr, xt, yt = load_adult(seed=42)
datasets['UCI Adult'] = (xp, yp, xpr, ypr, xt, yt, xp.shape[1])

# --- MNIST (784-D) ---
xp, yp, xpr, ypr, xt, yt = load_mnist(n_train_samples=5000, seed=42)
datasets['MNIST'] = (xp, yp, xpr, ypr, xt, yt, 784)

# --- CIFAR-10 features (512-D) ---
# RE-LOAD to ensure we have the correct 512-D variables and avoid overwriting bugs
print("\nRe-loading CIFAR-10 features for cross-dataset evaluation...")
xp, yp, xpr, ypr, xt, yt = load_cifar10_features(n_train_samples=5000, seed=42)
datasets['CIFAR-10 (features)'] = (xp, yp, xpr, ypr, xt, yt, 512)

results = []
for name, (xp, yp, xpr, ypr, xt, yt, dim) in datasets.items():
    print(f"\n{'='*50}")
    print(f"Dataset: {name}  (d={dim})")
    print(f"{'='*50}")

    # DP-GbC
    gbc_accs = []
    for s in SEEDS[:N_SEEDS_DS]:
        set_seed(s)
        # Adapt min_samples to dataset size
        ms = max(50, min(800, len(xp) // 4))
        _, acc, _, _, k = run_dpgbc(xp, yp, xt, yt,
                                    EPSILON, DELTA,
                                    theta=0.1, min_samples=ms,
                                    max_depth=50, tau=TAU)
        gbc_accs.append(acc)
    gbc_mean = np.mean(gbc_accs)

    # DP-SGD
    sgd_accs = []
    bs = min(256, len(xpr))
    ep = max(1, 4)  # keep 4 epochs
    for s in SEEDS[:N_SEEDS_DS]:
        acc, eps_act, _ = train_dpsgd(xpr, ypr, xt, yt,
                                      target_epsilon=EPSILON, delta=DELTA,
                                      batch_size=bs, epochs=ep,
                                      lr=0.1, momentum=0.9,
                                      max_grad_norm=1.0, seed=s)
        sgd_accs.append(acc)
    sgd_mean = np.mean(sgd_accs)

    results.append({
        'Dataset': name,
        'Dimension': dim,
        'DP-GbC Acc (%)': gbc_mean * 100,
        'DP-SGD Acc (%)': sgd_mean * 100
    })

    print(f"  DP-GbC Mean: {gbc_mean*100:.2f}%")
    print(f"  DP-SGD Mean: {sgd_mean*100:.2f}%")

# Summary Table
print("\n" + "=" * 72)
print("TABLE 4 SUMMARY")
print("=" * 72)
print(f"{'Dataset':<25} {'Dim':<8} {'DP-GbC Acc (%)':<18} {'DP-SGD Acc (%)':<18}")
print("-" * 72)
for r in results:
    print(f"{r['Dataset']:<25} {r['Dimension']:<8} {r['DP-GbC Acc (%)']:<18.2f} {r['DP-SGD Acc (%)']:<18.2f}")

In [ ]:
# ============================================================
# Cell 19: Hybrid Approach — Ball-Distance Features (Section 5.6)
# FIX: Re-load CIFAR-10 to avoid MNIST namespace collision from Cell 18
# ============================================================
print("=" * 70)
print("HYBRID APPROACH: Ball-Distance Features for DP-SGD (CIFAR-10)")
print("=" * 70)

set_seed(42)

# FIX: Explicitly re-load CIFAR-10 to ensure all variables are strictly 512-D
X_pub, y_pub, X_priv, y_priv, X_test, y_test = load_cifar10_features(
    n_train_samples=5000, seed=42
)

# Step 1: Generate coarse balls (m=800 -> k=2 balls)
print("Generating 2 coarse granular balls on public data...")
balls_hybrid = generate_granular_balls(X_pub, y_pub,
                                      theta=0.1, min_samples=800,
                                      max_depth=50)
k = len(balls_hybrid)
print(f"Number of balls (k): {k}")

# Step 2: Apply DP noise to the balls
private_balls_hybrid = apply_dp_noise(balls_hybrid, epsilon_total=5.0, delta_total=1e-5)

# Step 3: Compute distance-based features z = [||x - c_1||, ..., ||x - c_k||]
def compute_ball_distances(X, balls):
    """Transform data into k-dimensional distance space."""
    dists = np.empty((len(X), len(balls)))
    for j, ball in enumerate(balls):
        dists[:, j] = np.linalg.norm(X - ball.center, axis=1)
    return dists

print(f"Computing {k}-D ball-distance features...")
Z_priv = compute_ball_distances(X_priv, private_balls_hybrid)
Z_test = compute_ball_distances(X_test, private_balls_hybrid)

# Normalize distance features to [-1, 1] based on private set
zmin, zmax = Z_priv.min(axis=0), Z_priv.max(axis=0)
zrange = zmax - zmin
zrange[zrange == 0] = 1.0
Z_priv_n = 2.0 * (Z_priv - zmin) / zrange - 1.0
Z_test_n = 2.0 * (Z_test - zmin) / zrange - 1.0

# Step 4: Train DP-SGD on the k-D distance features
print(f"Training DP-SGD on {k}-D hybrid features (target ε=5.0)...")
hybrid_acc, hybrid_eps, _ = train_dpsgd(
    Z_priv_n, y_priv, Z_test_n, y_test,
    target_epsilon=5.0, delta=1e-5,
    batch_size=256, epochs=4,
    lr=0.1, momentum=0.9, max_grad_norm=1.0, seed=42
)

# Baseline: DP-SGD on the raw 512-D features for direct comparison
print("Training DP-SGD on raw 512-D features for comparison...")
raw_acc, raw_eps, _ = train_dpsgd(
    X_priv, y_priv, X_test, y_test,
    target_epsilon=5.0, delta=1e-5,
    batch_size=256, epochs=4,
    lr=0.1, momentum=0.9, max_grad_norm=1.0, seed=42
)

print("\n" + "-" * 50)
print(f"Hybrid DP-SGD Accuracy ({k}-D dist): {hybrid_acc*100:.2f}% (ε={hybrid_eps:.2f})")
print(f"Raw DP-SGD Accuracy (512-D):         {raw_acc*100:.2f}% (ε={raw_eps:.2f})")
print("-" * 50)

In [ ]:
# ============================================================
# Cell 20: Dimensionality Transition Plot (Figure 3)
# ============================================================
import matplotlib.ticker as ticker

plt.figure(figsize=(8, 5))

# Extract data from the results computed in Cell 18
dims = [r['Dimension'] for r in results]
dpgbc_accs = [r['DP-GbC Acc (%)'] for r in results]
dpsgd_accs = [r['DP-SGD Acc (%)'] for r in results]
names = [r['Dataset'] for r in results]

# Plot lines
plt.plot(dims, dpgbc_accs, 'o-', color='#d62728', label='DP-GbC (Geometric Noise)', linewidth=2.5, markersize=10)
plt.plot(dims, dpsgd_accs, 's--', color='#1f77b4', label='DP-SGD (Gradient Noise)', linewidth=2.5, markersize=10)

# Annotate points
for i, name in enumerate(names):
    offset_y_gbc = 3 if dpgbc_accs[i] < 80 else -6
    offset_y_sgd = -5 if dpsgd_accs[i] > 20 else 3
    plt.annotate(name, (dims[i], dpgbc_accs[i]),
                 textcoords="offset points", xytext=(10, offset_y_gbc), fontsize=9, color='#d62728')
    plt.annotate(name, (dims[i], dpsgd_accs[i]),
                 textcoords="offset points", xytext=(10, offset_y_sgd), fontsize=9, color='#1f77b4')

plt.xscale('log')
plt.xlabel('Data Dimensionality (log scale)', fontsize=12)
plt.ylabel('Test Accuracy (%)', fontsize=12)
plt.title('DP-GbC vs. DP-SGD: A Sharp Dimensionality Transition', fontsize=14)
plt.legend(fontsize=11, loc='lower left')
plt.grid(True, alpha=0.3, which="both")

# Set x-axis ticks to exact dimensions
plt.gca().xaxis.set_major_locator(ticker.LogLocator(base=10, numticks=10))
plt.gca().set_xticks([2, 100, 512, 784])
plt.gca().get_xaxis().set_major_formatter(ticker.ScalarFormatter())

plt.xlim(1, 1500)
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

print("Figure 3 generated: Notice the sharp collapse of DP-GbC accuracy as dimensionality increases beyond ~100.")